# 第4章：BPE 子词分词

---

## 1. OOV 问题的核心挑战

**OOV（Out-of-Vocabulary，词汇库外）问题**是自然语言处理（NLP）的核心痛点，指模型在推理或生成过程中遇到 **训练数据中未出现的新词或生僻词**，导致无法正确识别或处理这些词汇的问题。

### **典型场景**
- **新词涌现**：网络热词（如"内卷""躺平"）、专业术语（如"元宇宙"）等不断产生；
- **低频词问题**：训练语料中低频词（如"unhappiness"）未被纳入词表；
- **形态多样性**：词形变化（如"run""running""runner"）增加词表复杂度；
- **多语言差异**：通用词表难以覆盖不同语言的构词规则。

## 2. 传统分词方案的局限性

| **分词方案** | **优点** | **缺点** | **适用场景** |
|---|---|---|---|
| **字符级** | 词汇表极小，无未登录词（OOV） | 序列过长（语义单位破碎），模型学习效率低 | 小语种、低资源语言 |
| **词级** | 语义单位完整，人类可解释性强 | 词汇表极大，OOV 严重 | 简单任务、高资源语言 |
| **子词级（如 BPE）** | 词汇表大小适中，OOV 少 | 需预先训练分词器 | 预训练模型（BERT/GPT）、多语言任务 |

## 3. BPE 的核心原理

BPE（Byte Pair Encoding）通过 **迭代合并高频字符对**，逐步生成子词单元，平衡词汇表大小与语义表达精度：

1. **初始单位**：以字符为最小单位（如英文初始词汇表为所有字母、数字、标点符号）；
2. **迭代合并**：扫描文本，统计相邻字符对的频率，合并最高频的对；
3. **终止条件**：达到预设的词汇表大小或合并次数阈值。

## 4. BPE 与 Zipf 定律的关系

**Zipf 定律**：自然语言中，词频与词的排序满足幂律分布：
$$
\text{频率} \propto \frac{1}{\text{词序}}
$$

![Zipf Law示意图](img/4_1_zipfs-law.png)

BPE 利用 Zipf 定律：高频词优先合并快速构建常用词，低频词通过子词单元拆分减少 OOV。

## 5. BPE 执行流程

### 步骤 1：准备训练数据
训练文本：
```
"low low low lowly lower newer newer"
```

### 步骤 2：统计子词对频率

### 步骤 3：迭代合并子词对
1. **第一次合并**：`(l,o)` → `lo`
2. **第二次合并**：`(lo,w)` → `low`
3. **第三次合并**：`(low,</w>)` → `low</w>`
4. **第四次合并**：`(e,r)` → `er`

## 6. BPE 解码过程

BPE 解码是分词的逆过程：
1. **直接拼接**所有子词；
2. **遇到终止符 `</w>`** 时，替换为空格；
3. **去除多余空格**，得到原始句子。

## 7. BPE 优缺点总结

**优点**：控制词汇表大小、减少 OOV、语义连贯性、多语言适配

**缺点**：依赖训练数据、固定合并规则、处理效率

## 8. BPE 代码实现
### 8.1 学习 BPE 规则（learn_bpe）

In [ ]:
from __future__ import unicode_literals
import os
import sys
import inspect
import codecs
import re
import copy
import warnings
from collections import defaultdict, Counter

In [ ]:
def update_vocabulary(vocab, file_name, is_dict=False):
    """
    统计文本文件中的词汇频率，更新词汇表字典。

    参数:
        vocab (dict 或 defaultdict(int)): 待更新的词汇表。若为普通字典，新单词默认计数为1。
        file_name (str): 输入文件路径。
        is_dict (bool): 若为 True，每行格式为 "word count"；若为 False，每行按空白分割单词。

    返回:
        defaultdict(int): 更新后的词汇表（键：单词，值：频率计数）。
    """
    if not isinstance(vocab, defaultdict):
        vocab = defaultdict(int, vocab)
    with open(file_name, 'r', encoding='utf-8-sig') as fobj:
        for line_num, line in enumerate(fobj, 1):
            line = line.strip('\r\n')
            if not line:
                continue
            if is_dict:
                parts = line.split()
                if len(parts) != 2:
                    print(f"警告：第 {line_num} 行格式错误（预期 'word count'）: {line}")
                    continue
                word, count_str = parts
                try:
                    count = int(count_str)
                except ValueError as e:
                    print(f"警告：第 {line_num} 行计数转换失败: {count_str}。错误: {e}")
                    continue
                vocab[word] += count
            else:
                words = line.split()
                for word in words:
                    vocab[word] += 1
    return vocab


def get_pair_statistic(vocab):
    """
    统计符号对频率和位置
    :param vocab: 词汇表
    :return: stats（符号对频率字典）, indices（符号对索引字典）
    """
    stats = defaultdict(int)
    indices = defaultdict(lambda: defaultdict(int))
    for i, (word, freq) in enumerate(vocab):
        prev_char = word[0]
        for char in word[1:]:
            stats[prev_char, char] += freq
            indices[prev_char, char][i] += 1
            prev_char = char
    return stats, indices

In [ ]:
def update_pair_statistics(pair, changed, stats, indices):
    """
    更新符号对的索引和频率
    :param pair: (str,str) 当前要合并的符号对
    :param changed: List[(j,word,old_word,freq)] 受影响的单词列表
    :param stats: defaultdict(int) 符号对频率字典
    :param indices: 符号对索引字典
    """
    stats[pair] = 0
    indices[pair] = defaultdict(int)
    first, second = pair
    new_pair = first + second

    for j, word, old_word, freq in changed:
        i = 0
        try:
            i = old_word.index(first, i)
        except ValueError as e:
            break
        if i < len(old_word) - 1 and old_word[i + 1] == second:
            if i:
                prev = old_word[i - 1:i + 1]
                stats[prev] -= freq
                indices[prev][j] -= 1
            if i < len(old_word) - 2:
                if old_word[i + 2] != first or i >= len(old_word) - 3 or old_word[i + 3] != second:
                    nex = old_word[i + 1:i + 3]
                    stats[nex] -= freq
                    indices[nex][j] -= 1
            i += 2
        else:
            i += 1

        i = 0
        while True:
            try:
                i = word.index(new_pair, i)
            except ValueError as e:
                break
            if i:
                prev = word[i - 1, i + 1]
                stats[prev] += freq
                indices[prev][j] += 1
            if i < len(word) - 1 and word[i + 1] != new_pair:
                nex = word[i:i + 2]
                stats[nex] += freq
                indices[nex][j] += 1
            i += 1

In [ ]:
def replace_pair(pair, vocab, indices):
    """
    用于将词汇表中所有指定符号对（如 ('A', 'B')）的出现替换为合并后的新符号（如 AB）
    :param pair: (str,str) 待合并的符号对
    :param vocab: List((word,freq))
    :param indices: {key:pair,val:{idx,freq}}
    :return: 记录所有被修改的单词信息
    """
    first, second = pair
    pair_str = ''.join(pair)
    pair_str = pair_str.replace('\\', '\\\\')
    pattern = re.compile(r'(?<!\S)' + re.escape(first + ' ' + second) + r'(?!\S)')
    iterator = indices[pair].items()
    changes = []
    for j, freq in iterator:
        if freq < 1:
            continue
        word, freq = vocab[j]
        new_word = ''.join(word)
        new_word = pattern.sub(pair_str, new_word)
        new_word = tuple(new_word.split(' '))
        vocab[j] = (new_word, freq)
        changes.append((j, new_word, word, freq))
    return changes


def prune_stats(stats, big_stats, threshold):
    """
    通过删除频率低于阈值的符号对，减小 stats字典的规模
    """
    for item, freq in list(stats.items()):
        if freq < threshold:
            del stats[item]
        if freq < 0:
            big_stats[item] += freq
        else:
            big_stats[item] = big_stats[item]

In [ ]:
def learn_bpe(infile_names, outfile_name, num_symbols, min_frequency=2, verbose=False, is_dict=False,
              total_symbols=False):
    """
    学习 BPE 子词合并规则的主函数

    :param infile_names: List[str] 输入文件路径列表
    :param outfile_name: str 输出文件路径
    :param num_symbols: int 需要学习的子词数量
    :param min_frequency: int 符号对的最小频率阈值
    :param verbose: bool 是否启用详细输出模式
    :param is_dict: bool 输入文件是否为"字典格式"
    :param total_symbols: bool 是否将词内字符和词尾字符视为独立子词
    """
    sys.stderr = codecs.getwriter('UTF-8')(sys.stderr.buffer)
    sys.stdout = codecs.getwriter('UTF-8')(sys.stdout.buffer)
    sys.stdin = codecs.getreader('UTF-8')(sys.stdin.buffer)

    vocab = Counter()
    for f in infile_names:
        sys.stderr.write('Collexting vocab from {}\n'.format(f))
        vocab = update_vocabulary(vocab, f, is_dict)
    vocab = dict([(tuple(x[:-1]) + (x[-1] + '</w>',), y) for (x, y) in vocab.items()])
    sorted_vocab = sorted(vocab.items(), key=lambda x: x[1], reverse=True)

    stats, indices = get_pair_statistic(sorted_vocab)
    big_stats = copy.deepcopy(stats)

    if total_symbols:
        uniq_char_internal = set()
        uniq_char_final = set()
        for word in vocab:
            for char in word[:-1]:
                uniq_char_internal.add(char)
            uniq_char_final.add(word[-1])
        num_symbols -= len(uniq_char_internal) + len(uniq_char_final)

    sys.stderr.write(f'Write vocab file to {outfile_name}')
    with codecs.open(outfile_name, 'w', encoding='utf-8') as outfile:
        outfile.write('#version: 0.2\n')
        threshold = max(stats.values()) / 10
        for i in range(num_symbols):
            most_frequent = max(stats, key=lambda x: (stats[x], x))
            if not stats or (i and stats[most_frequent] < threshold):
                prune_stats(stats, big_stats, threshold)
                stats = copy.deepcopy(big_stats)
                most_frequent = max(stats, key=lambda x: (stats[x], x))
                threshold = stats[most_frequent] * i / (i + 10000.0)
                prune_stats(stats, big_stats, threshold)

            if stats[most_frequent] < min_frequency:
                sys.stderr.write(f'no pair has frequency >= {min_frequency}. Stopping\n')
                break

            if verbose:
                sys.stderr.write(f'pair{i}:{most_frequent[0]} {most_frequent[1]}  freq{stats[most_frequent]}\n')
            outfile.write(f'{most_frequent[0]} {most_frequent[1]}\n')
            changes = replace_pair(most_frequent, sorted_vocab, indices)
            update_pair_statistics(most_frequent, changes, stats, indices)
            stats[most_frequent] = 0
            if not i % 100:
                prune_stats(stats, big_stats, threshold)

### 8.2 BPE 编码与解码（BPE 类）

In [ ]:
from __future__ import unicode_literals, division
import sys
import os
import inspect
import codecs
import io
import re
import warnings
import random

In [ ]:
class BPE(object):
    """
    BPE 分词器：加载 BPE 规则并应用于新文本的编码与解码
    """

    def __init__(self, codes, merges=-1, separator='@@', vocab=None, glossaries=None):
        """
        :param codes: BPE规则文件对象
        :param merges: 合并次数限制
        :param separator: 子词分隔符（如 @@）
        :param vocab: 预定义词汇表
        :param glossaries: 用户指定的术语列表
        """
        codes.seek(0)
        offset = 1

        firstline = codes.readline()
        if firstline.startswith('#version:'):
            self.version = tuple([int(x) for x in re.sub(r'(\.0+)*$', '', firstline.split()[-1]).split(".")])
            offset += 1
        else:
            self.version = (0, 1)
        self.bpe_codes = [tuple(item.strip('\r\n').split(' '))
                          for (n, item) in enumerate(codes)
                          if (n < merges or merges == -1)]
        for i, item in enumerate(self.bpe_codes):
            if len(item) != 2:
                if len(item) != 2:
                    sys.stderr.write(f'Error: invalid line {i + offset} in BPE codes file: {" ".join(item)}\n')
                    sys.stderr.write('The line should exist of exactly two subword units, separated by whitespace\n')
                    sys.exit(1)
            codes.seek(0)
        self.bpe_codes = dict([(code, i) for (i, code) in reversed(list(enumerate(self.bpe_codes)))])
        self.bpe_codes_reverse = dict([(pair[0] + pair[1], pair) for pair, i in self.bpe_codes.items()])
        self.separator = separator
        self.vocab = vocab
        self.glossaries = glossaries if glossaries else []
        self.glossaries_regex = re.compile('^({})$'.format('|'.join(self.separator))) if glossaries else None
        self.cache = {}

    def process_line(self, line, dropout=0):
        """处理单行文本：保留前导/尾随空白符，对中间内容进行 BPE 分词"""
        out = ""
        leading_whitespace = len(line) - len(line.lstrip('\r\n '))
        if leading_whitespace:
            out += line[:leading_whitespace]
        out += self.segment(line, dropout)
        trailing_whitespace = len(line) - len(line.rstrip('\r\n '))
        if trailing_whitespace and trailing_whitespace != len(line):
            out += line[-trailing_whitespace:]
        return out

    def segment(self, sentence, dropout=0):
        """处理完整的文本行，按空格分词后调用 segment_tokens"""
        segments = self.segment_tokens(sentence.strip('\r\n').split(' '), dropout)
        return ' '.join(segments)

    def segment_tokens(self, tokens, dropout=0):
        """对每个 token 执行 BPE 编码"""
        output = []
        for word in tokens:
            if not word:
                continue
            new_word = [out for segment in self._isolate_glossaries(word)
                        for out in encode(segment,
                                          self.bpe_codes,
                                          self.bpe_codes_reverse,
                                          self.vocab,
                                          self.separator,
                                          self.version,
                                          self.cache,
                                          self.glossaries_regex,
                                          dropout
                                          )]
            for item in new_word[:-1]:
                output.append(item + self.separator)
            output.append(new_word[-1])
            return output

    def _isolate_glossary(self, word):
        """隔离单词中包含的术语"""
        word_segments = [word]
        for gloss in self.glossaries:
            word_segments = [out_segments for segment in word_segments
                             for out_segments in isolate_glossary(segment, gloss)]
        return word_segments

In [ ]:
def encode(orig, bpe_codes, bpe_codes_reverse, vocab, separator, version, cache, glossaries_regex=None, dropout=0):
    """
    输入的原始单词（orig）通过应用 BPE 合并规则，转换为符合词汇表的子词序列

    :param orig: 待编码的原始单词
    :param bpe_codes: 正向合并规则字典
    :param bpe_codes_reverse: 反向合并规则字典
    :param vocab: 预定义词汇表
    :param separator: 子词分隔符
    :param version: BPE 版本
    :param cache: 缓存字典
    :param glossaries_regex: 术语正则表达式
    :param dropout: 训练时随机丢弃合并对的概率
    :return: 编码后的子词元组
    """
    if not dropout and orig in cache:
        return cache[orig]
    if glossaries_regex and glossaries_regex.match(orig):
        cache[orig] = (orig,)
        return (orig,)

    if version == (0, 1):
        word = list(orig) + ['</w>']
    elif version == (0, 2):
        word = list(orig[:-1]) + [orig[-1] + '</w>']

    while len(word) > 1:
        pairs = [
            (bpe_codes[pair], i, pair)
            for (i, pair) in enumerate(zip(word, word[1:]))
            if (not dropout or random.random() > dropout)
               and pair in bpe_codes
        ]
        if not pairs:
            break

        bigram = min(pairs)[2]
        positions = [i for (rank, i, pair) in pairs if pair == bigram]

        i = 0
        new_word = []
        bigram_str = ''.join(bigram)
        for j in positions:
            if j < i:
                continue
            new_word.extend(word[i:j])
            new_word.append(bigram_str)
            i = j + 2
        new_word.extend(word[i:])
        word = new_word

        if word[-1] == '</w>':
            word = word[:-1]
        elif word[-1].endswith('</w>'):
            word[-1] = word[-1][:-4]

        word = tuple(word)
        if vocab:
            word = check_vocab_and_split(word, bpe_codes_reverse, vocab, separator)

        cache[orig] = word
        return word


def recursive_split(segment, bpe_codes_reverse, vocab, separator, final=False):
    """
    反向拆分 OOV 子词：通过反转 BPE 合并规则，逐步拆分为更小的子单元
    """
    try:
        if final:
            left, right = bpe_codes_reverse[segment + '</w>']
            right = right[:-4]
        else:
            left, right = bpe_codes_reverse[segment]
    except KeyError:
        yield segment
        return

    if left + separator in vocab:
        yield left
    else:
        for item in recursive_split(left, bpe_codes_reverse, vocab, separator, final=False):
            yield item

    if (final and right in vocab) or (not final and right + separator in vocab):
        yield right
    else:
        for item in recursive_split(right, bpe_codes_reverse, vocab, separator, final=True):
            yield item


def check_vocab_and_split(orig, bpe_codes_reverse, vocab, separator):
    """
    检查子词是否在词汇表中，若不在则递归拆分
    """
    out = []
    for segment in orig[:-1]:
        if segment + separator in vocab:
            out.append(segment)
        else:
            for item in recursive_split(segment, bpe_codes_reverse, vocab, separator, False):
                out.append(item)

    segment = orig[-1]
    if segment in vocab:
        out.append(segment)
    else:
        for item in recursive_split(segment, bpe_codes_reverse, vocab, separator, True):
            out.append(item)
    return out


def read_vocabulary(vocab_file, threshold):
    """
    读取词汇表文件，保留满足频率阈值的单词
    """
    vocabulary = set()
    for line in vocab_file:
        word, freq = line.strip('\r\n').split(' ')
        freq = int(freq)
        if threshold == None or freq >= threshold:
            vocabulary.add(word)
    return vocabulary


def isolate_glossary(self, word, glossary):
    """
    隔离单词中包含的术语
    :param self: 待处理的原始单词
    :param word: 需要隔离的术语
    :return:
    """
    if re.match('^' + glossary + '$', word) or not re.search(glossary, word):
        return [word]
    else:
        segments = re.split(r'({})'.format(glossary), word)
        segments, ending = segments[:-1], segments[-1]
        segments = list(filter(None, segments))
        return segments + [ending.strip('\r\n ')] if ending != '' else segments

# 04 BPE 分词 ── 子词级别分词
## 什么是 BPE？
BPE = Byte-Pair Encoding（字节对编码）
从字母开始，逐步合并高频出现的字符对。

In [ ]:
from collections import defaultdict
texts = ['low','lower','lowest','high','higher','new','newer','newest']
vocab = defaultdict(int)
for t in texts:
    chars = ' '.join(list(t)) + ' </w>'
    vocab[chars] += 1
    print(f'{t:8} -> {chars}')


low      -> l o w </w>
lower    -> l o w e r </w>
lowest   -> l o w e s t </w>
high     -> h i g h </w>
higher   -> h i g h e r </w>
new      -> n e w </w>
newer    -> n e w e r </w>
newest   -> n e w e s t </w>


In [ ]:
def get_stats(v):
    p = defaultdict(int)
    for w,f in v.items():
        s = w.split()
        for i in range(len(s)-1): p[(s[i],s[i+1])] += f
    return p
pairs = get_stats(vocab)
print('字符对频率:')
for pair,f in sorted(pairs.items(),key=lambda x:-x[1]):
    print(f'  {pair[0]}+{pair[1]} -> {f}次')
best = max(pairs,key=pairs.get)
print(f'最高频: "{best[0]}"+"{best[1]}" -> "{best[0]+best[1]}"')


字符对频率:
  w+e -> 4次
  l+o -> 3次
  o+w -> 3次
  e+r -> 3次
  r+</w> -> 3次
  n+e -> 3次
  e+w -> 3次
  w+</w> -> 2次
  e+s -> 2次
  s+t -> 2次
  t+</w> -> 2次
  h+i -> 2次
  i+g -> 2次
  g+h -> 2次
  h+</w> -> 1次
  h+e -> 1次
最高频: "w"+"e" -> "we"


In [ ]:
def merge_vocab(pair,v):
    bg=' '.join(pair);rep=''.join(pair)
    return {w.replace(bg,rep):f for w,f in v.items()}
vocab2 = merge_vocab(best, vocab)
print('合并后:')
for w,f in sorted(vocab2.items(),key=lambda x:-x[1]):
    print(f'  "{w}" ({f}次)')


合并后:
  "l o w </w>" (1次)
  "l o we r </w>" (1次)
  "l o we s t </w>" (1次)
  "h i g h </w>" (1次)
  "h i g h e r </w>" (1次)
  "n e w </w>" (1次)
  "n e we r </w>" (1次)
  "n e we s t </w>" (1次)


In [ ]:
def learn_bpe(texts,nm=50):
    v = defaultdict(int)
    for t in texts: v[' '.join(list(t))+' </w>'] += 1
    merges = []
    for _ in range(nm):
        p = get_stats(v)
        if not p: break
        b = max(p,key=p.get); merges.append(b); v = merge_vocab(b,v)
    return merges
merges = learn_bpe(texts,10)
for i,(a,b) in enumerate(merges,1):
    print(f'#{i}: "{a}"+"{b}" -> "{a+b}"')


#1: "w"+"e" -> "we"
#2: "l"+"o" -> "lo"
#3: "r"+"</w>" -> "r</w>"
#4: "n"+"e" -> "ne"
#5: "w"+"</w>" -> "w</w>"
#6: "lo"+"we" -> "lowe"
#7: "s"+"t" -> "st"
#8: "st"+"</w>" -> "st</w>"
#9: "h"+"i" -> "hi"
#10: "hi"+"g" -> "hig"
